# Day 5 — Unsupervised learning

- PCA + KMeans on MNIST (image data)
- Choosing K with elbow + silhouette
- KMeans segmentation on Mall Customers (interpretable clusters)


## 1. MNIST — PCA visualization + KMeans

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np, matplotlib.pyplot as plt

mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data, mnist.target.astype(int)
# Subsample for speed
rng = np.random.default_rng(0)
idx = rng.choice(len(X), size=10_000, replace=False)
Xs, ys = X[idx], y[idx]
Xs = StandardScaler().fit_transform(Xs)
Xs.shape


In [ ]:
pca50 = PCA(n_components=50, random_state=0).fit_transform(Xs)
pca2  = PCA(n_components=2,  random_state=0).fit_transform(Xs)
plt.figure(figsize=(8, 6))
plt.scatter(pca2[:, 0], pca2[:, 1], c=ys, cmap='tab10', s=4, alpha=0.7)
plt.title('MNIST in 2D via PCA'); plt.colorbar(label='digit');


In [ ]:
km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(pca50)
labels = km.labels_

# Cluster purity vs true labels
from collections import Counter
purity = sum(max(Counter(ys[labels == k]).values()) for k in range(10)) / len(ys)
print(f'cluster purity = {purity:.3f}')


## 2. Choosing K — elbow + silhouette

In [ ]:
from sklearn.metrics import silhouette_score
ks = range(2, 12)
inertias, sils = [], []
for k in ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(pca50)
    inertias.append(m.inertia_)
    sils.append(silhouette_score(pca50[:2000], m.labels_[:2000]))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), inertias, 'o-'); ax[0].set_title('Elbow (inertia)'); ax[0].set_xlabel('k')
ax[1].plot(list(ks), sils, 'o-');     ax[1].set_title('Silhouette');      ax[1].set_xlabel('k')


## 3. Mall Customers — interpretable segmentation

Download `Mall_Customers.csv` from https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python and place it next to this notebook.

In [ ]:
import pandas as pd
from pathlib import Path

p = Path('Mall_Customers.csv')
if not p.exists():
    print('Download Mall_Customers.csv first; skipping this section.')
else:
    mc = pd.read_csv(p)
    print(mc.head())
    feats = mc[['Annual Income (k$)', 'Spending Score (1-100)']].values
    km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(feats)
    plt.figure(figsize=(7, 5))
    plt.scatter(feats[:, 0], feats[:, 1], c=km.labels_, cmap='tab10', s=40)
    plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], c='black', marker='X', s=200)
    plt.xlabel('Annual income (k$)'); plt.ylabel('Spending score'); plt.title('Mall customer segments');


### Exercises
- Try `DBSCAN` on the Mall data and compare segments.
- Replace PCA with UMAP (`pip install umap-learn`) on MNIST and re-plot.
- Use cluster labels as a feature in a downstream classifier — does it help?
